In [1]:
# Install if needed:
# !pip install pandas scikit-learn nltk matplotlib

import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

nltk.download("stopwords")

# 1. Load dataset
df = pd.read_csv("precision_multilabel.csv")
print("Shape:", df.shape)
print(df.head())

# 2. Select text column
# Change "text" if your CSV uses another column name
text_col = "text"

# 3. NLP cleaning
stop_words = set(stopwords.words("english"))

def clean_text(x):
    x = str(x).lower()
    x = re.sub(r"http\S+|[^a-zA-Z\s]", " ", x)
    x = " ".join(w for w in x.split() if w not in stop_words)
    return x

df["clean_text"] = df[text_col].apply(clean_text)

# 4. TF-IDF
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = tfidf.fit_transform(df["clean_text"])

# 5. Find best K
scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    scores[k] = silhouette_score(X, labels)

best_k = max(scores, key=scores.get)
print("Best K:", best_k)
print("Silhouette Score:", scores[best_k])

# 6. K-Means clustering
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# 7. Show results
print("\nCluster counts:")
print(df["cluster"].value_counts())

display(df[[text_col, "cluster"]].head(20))

# 8. Important words per cluster
terms = tfidf.get_feature_names_out()

for i in range(best_k):
    top_words = kmeans.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"Cluster {i}:", ", ".join(terms[top_words]))

# 9. Save results
df.to_csv("precision_multilabel_kmeans.csv", index=False)

print("\nSaved as precision_multilabel_kmeans.csv")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Shape: (569628, 1)
   8.617316716144302058e-02
0                  0.086172
1                  0.086172
2                  0.086172
3                  0.086172
4                  0.086172


KeyError: 'text'